# RQ1 — one-class model comparison, evaluated without any background (sensitivity only)

**Purpose.** The orientador asked, in the meeting of 2026-11-17, to
reopen the one-class comparison of
[`one_class_method_selection.ipynb`](one_class_method_selection.ipynb):
that notebook trained One-Class SVM and Isolation Forest only on
presences, but still evaluated them with ROC-AUC and the continuous
Boyce index against the background pool — so the evaluation kept the
assumption ("the background has no acacia") that a true one-class
method is meant to avoid. Full decision, reasoning and the
implementation choices below (reuse the existing block grid instead of
a new 4x4 grid, PCA as the autoencoder, contamination/threshold,
ensemble rule): [`thesis/notes.md`](../../thesis/notes.md#one-class-reaberto-avaliação-sem-fundo-sensitivity-grelha-espacial-e-ensemble-pca--isolation-forest).

**What changes from `one_class_method_selection.ipynb`:**
- No background point enters this notebook anywhere — not in
  training (already true there) and not in evaluation (new).
- The metric is sensitivity on held-out presences, not ROC-AUC/Boyce.
- The spatial split is leave-one-block-out over the existing 10 km
  blocks, not a random 20%-of-blocks holdout repeated over 5 seeds.
- Two more models are added: PCA reconstruction error (the autoencoder
  the orientador asked for, in its simplest/linear form — Baldi &
  Hornik, 1989) and an ensemble of PCA + Isolation Forest.

**Scope:** citizen-science *A. dealbata* presences only (the arm with
"only presences, little data" that motivated the request). The expert
arm is not redone.


## 1. Presence embeddings (citizen science)

Same loading as `one_class_method_selection.ipynb` — the cleaned
iNaturalist presences joined to their TESSERA pilot embedding by pixel
id. No background pool is loaded in this notebook.


In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from src.embeddings import block_id, load_pilot_embeddings, match_to_pixel

EMB_DIR = ROOT / "data" / "processed" / "embeddings"
PILOT_DIR = EMB_DIR / "tessera_v1.1_2023_rq1_pilot"

pilot_df, pilot_emb = load_pilot_embeddings(PILOT_DIR)

dealbata = pd.read_csv(ROOT / "data" / "interim" / "citizen_science" / "inat_dealbata_clean.csv")
pres_x, pres_y, pres_emb, n_matched = match_to_pixel(dealbata["pixel"], pilot_df, pilot_emb)
print(f"presences matched to an embedding: {n_matched}/{len(dealbata)}")
print(f"presences with a non-NaN embedding: {len(pres_emb)}")


## 2. Spatial blocks: leave-one-block-out

Reuses the same 10 km UTM block grid as `one_class_method_selection.ipynb`
(`src.embeddings.block_id`) — simpler than building a new 4x4 grid, and
the existing grid already gives a comparable number of cells (below).
Every block with at least 5 presences is used once as the test fold
(the rest train); blocks with fewer than 5 presences are too small to
report a meaningful sensitivity and are left out of the fold list
(same 5-point floor as the presence/background counts in
`one_class_method_selection.ipynb`).


In [ ]:
pres_block = block_id(pres_x, pres_y)
block_counts = pd.Series(pres_block).value_counts().sort_values(ascending=False)
print(f"{len(block_counts)} blocks hold at least one presence")

test_blocks = sorted(block_counts[block_counts >= 5].index)
print(f"{len(test_blocks)} blocks qualify as a leave-one-block-out test fold (>= 5 presences)")
print("presences per qualifying block:", sorted(block_counts[test_blocks].tolist()))


## 3. Models, training and threshold — no background anywhere

Four scores, each trained only on the training-fold presences (scaled
with a `StandardScaler` fit on that fold's training presences):

- **One-Class SVM** (`nu=0.10`) and **Isolation Forest**
  (`contamination=0.10`) — same algorithms as
  `one_class_method_selection.ipynb`, same contamination/`nu` (one of
  the values already swept there), now with a different evaluation.
- **PCA reconstruction error** — the autoencoder the orientador asked
  for, in its simplest form: a linear autoencoder is mathematically a
  PCA projection and reconstruction (Baldi & Hornik, 1989). Components
  kept to explain 90% of the training presences' variance. Chosen
  over a neural autoencoder because it needs no architecture choice
  and no extra data to train well — with ~2,400 presences split over
  15 folds, a neural network would need more data to learn a useful
  reconstruction, which would work against the very reason given for
  suggesting an autoencoder (few data).
- **Ensemble (PCA + Isolation Forest)** — the pairing the orientador
  named. Combined by averaging each model's score after standardising
  it (z-score) on the training fold — the simplest combination, no
  weight to tune.

**Threshold, fixed before looking at any test fold:** for every model,
a presence is "recognised" if its score is at or above the 10th
percentile of that model's own **training**-fold scores (so each
model is calibrated to flag about 10% of its own training presences
as atypical, consistently across models). Sensitivity is the fraction
of a test fold's presences at or above that threshold. No background
point is used to set or check this threshold.

This also means a sensitivity near 0.90 is the expected value when a
held-out block looks like the training blocks (because the threshold
is fixed at the training 10th percentile); a lower score means the
held-out block is harder to generalise to, not that the metric itself
is degenerate.


In [ ]:
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM

CONTAMINATION = 0.10


def pca_anomaly_score(train_s, x_s):
    pca = PCA(n_components=0.90, svd_solver="full", random_state=0).fit(train_s)
    recon = pca.inverse_transform(pca.transform(x_s))
    error = ((x_s - recon) ** 2).sum(axis=1)
    return -error, pca.n_components_  # higher = more normal, like the other scores


results = {"OneClassSVM": [], "IsolationForest": [], "PCA": [], "Ensemble (PCA + IsolationForest)": []}
n_test_presences = []
n_pca_components = []

for test_block in test_blocks:
    train_mask = pres_block != test_block
    test_mask = pres_block == test_block
    train, test = pres_emb[train_mask], pres_emb[test_mask]
    n_test_presences.append(len(test))

    scaler = StandardScaler().fit(train)
    train_s, test_s = scaler.transform(train), scaler.transform(test)

    def sensitivity(train_score, test_score):
        threshold = np.percentile(train_score, 100 * CONTAMINATION)
        return (test_score >= threshold).mean()

    svm = OneClassSVM(kernel="rbf", nu=0.10, gamma="scale").fit(train_s)
    results["OneClassSVM"].append(sensitivity(svm.decision_function(train_s), svm.decision_function(test_s)))

    iso = IsolationForest(n_estimators=200, contamination=CONTAMINATION, random_state=42).fit(train_s)
    iso_train, iso_test = iso.decision_function(train_s), iso.decision_function(test_s)
    results["IsolationForest"].append(sensitivity(iso_train, iso_test))

    pca_train, n_comp = pca_anomaly_score(train_s, train_s)
    pca_test, _ = pca_anomaly_score(train_s, test_s)
    n_pca_components.append(n_comp)
    results["PCA"].append(sensitivity(pca_train, pca_test))

    z = lambda s, ref: (s - ref.mean()) / ref.std()
    ens_train = (z(iso_train, iso_train) + z(pca_train, pca_train)) / 2
    ens_test = (z(iso_test, iso_train) + z(pca_test, pca_train)) / 2
    results["Ensemble (PCA + IsolationForest)"].append(sensitivity(ens_train, ens_test))

summary = pd.DataFrame({
    "sensitivity (mean)": {m: np.mean(v) for m, v in results.items()},
    "sensitivity (sd)": {m: np.std(v) for m, v in results.items()},
    "n_folds": {m: len(v) for m, v in results.items()},
})
summary.round(3)


In [ ]:
print(f"test presences per fold: min {min(n_test_presences)}, max {max(n_test_presences)}, "
      f"over {len(test_blocks)} folds")
print(f"PCA components kept for 90% variance: min {min(n_pca_components)}, "
      f"max {max(n_pca_components)}, of {pres_emb.shape[1]} embedding dimensions")


## 4. Result — and why it does not rank the models

Sensitivity, mean (sd) over the 15 folds: Isolation Forest 0.916
(0.068); ensemble PCA + Isolation Forest 0.876 (0.119); One-Class SVM
0.849 (0.110); PCA 0.815 (0.158).

**These numbers do not say which model discriminates acacia from the
rest of the landscape.** With the threshold taken from each model's own
training 10th percentile, sensitivity only measures whether the
held-out presences look like the training presences. A model that gives
nearly the same score to every site also lands near 0.90 on every
block. Isolation Forest's higher value is therefore not evidence that
it generalises better — it was the model that separated presences from
the background worst in `one_class_method_selection.ipynb` (ROC-AUC
0.57), so its scores may simply be less informative. (First written
with the opposite reading on 2026-11-17 and corrected the same day; the
record is in `thesis/notes.md`.)

**What this run does show:** an evaluation that uses literally only
presences cannot rank models. Measuring discrimination needs the
landscape — but as a sample of available area, never as absence. The
design that follows from this (presence-only classifier chosen between
true one-class and presence-background methods, sensitivity at a fixed
fraction of predicted area, same leave-one-block-out folds as here):
[`thesis/notes.md`](../../thesis/notes.md#rq1-presence-only).

**Run details:** single untuned run, one contamination/`nu` value, no
nested validation; 15 folds with 7 to 819 test presences each; not
comparable to the ROC-AUC/Boyce numbers of
`one_class_method_selection.ipynb` (different metric, split and use of
background).
